# Project 1

I'll be exploring the University of Maryland's, available here: https://www.start.umd.edu/data-tools/GTD. This database includes instances of terrorism around the world from 1970 to 2020. The dataset includes a lot of information (120 variables!) on over 200,000 terrorist attacks. Most of this information I won't be looking at today, but it's really interesting stuff.

In [ ]:
# First, I need to load in the dataset, which is saved as a .csv file,
# using pandas, a module that contains a lot of useful functions for
# data analysis.

import pandas as pd

In [ ]:
# Now I'll load the dataset and save it as a variable, which I'll name
# tdb (for terrorism data base), so I can easily work with the data.
# Then I'll check that it works by calling the first five observations
# in the dataset.

tdb = pd.read_csv("globalterrorismdb_0522dist.csv")

tdb.head(5)

C:\Users\hurre\AppData\Local\Temp\ipykernel_27848\2119646036.py:5: DtypeWarning: Columns (4: approxdate, 31: attacktype2_txt, 33: attacktype3_txt, 54: corp3, 61: gsubname2, 62: gname3, 63: gsubname3, 76: claimmode2_txt, 79: claimmode3_txt, 90: weaptype3_txt, 92: weapsubtype3_txt, 94: weaptype4_txt, 96: weapsubtype4_txt, 114: divert, 115: kidhijcountry, 121: ransomnote) have mixed types. Specify dtype option on import or set low_memory=False.
  tdb = pd.read_csv("globalterrorismdb_0522dist.csv")


,eventid,iyear,imonth,iday,approxdate,extended,resolution,country,country_txt,region,...,addnotes,scite1,scite2,scite3,dbsource,INT_LOG,INT_IDEO,INT_MISC,INT_ANY,related
0,197000000001,1970,7,2,NaN,0,NaN,58,Dominican Republic,2,...,NaN,NaN,NaN,NaN,PGIS,0,0,0,0,NaN
1,197000000002,1970,0,0,NaN,0,NaN,130,Mexico,1,...,NaN,NaN,NaN,NaN,PGIS,0,1,1,1,NaN
2,197001000001,1970,1,0,NaN,0,NaN,160,Philippines,5,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
3,197001000002,1970,1,0,NaN,0,NaN,78,Greece,8,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
4,197001000003,1970,1,0,NaN,0,NaN,101,Japan,4,...,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN


I'm going to choose to look at one numerical variable, although there are lots. I chose the number of people killed in each attack (both civilians and attackers), which is nkill in the dataset. I'll use pandas functions to calculate the mean (average number), mode (which value appears most frequently), and median (the middle value in the ordered set) of fatalities.

In [ ]:
# due to the file size, I'm going to trim it to just the columns I need
# so it can sync to github

tdb[["iyear", "nkill"]].to_csv("terrorism_trimmed.csv", index=False)

In [34]:
tdb2 = pd.read_csv("terrorism_trimmed.csv")

In [35]:
# First let's see the average, or mean.

tdb2["nkill"].mean()

np.float64(2.4310296735453574)

In [22]:
# Now let's see the mode.

tdb2["nkill"].mode()

0    0.0
Name: nkill, dtype: float64

In [23]:
# And lastly, the median.

tdb2["nkill"].median()

np.float64(0.0)

### And now the hard way...

I'll calculate the same numbers---mean, mode, and median---without using pandas, and see if I get the same answers.

In [36]:
import csv

FILENAME = "terrorism_trimmed.csv"  # opening the dataset
COLUMN = "nkill"  # choosing the variable for number of fatalities

# 1. Read the column into a list of numbers
values = []
with open(
    FILENAME, newline="", encoding="utf-8"
) as f:  # note: I did use claude to figure out I
    reader = csv.DictReader(f)  # needed the utf-8 encoding
    for row in reader:
        cell = row[COLUMN].strip()
        if cell == "":
            continue  # skip missing values
        try:
            values.append(float(cell))
        except ValueError:
            continue  # skip non-numeric values like "N/A"

# 2. Mean: add everything up, divide by how many observations (i.e. terrorist attacks)
total = 0
count = 0
for v in values:
    total += v
    count += 1
mean_value = total / count

# 3. Median: sort, then take the middle value
sorted_vals = sorted(values)
mid = count // 2
if count % 2 == 1:
    median_value = sorted_vals[mid]
else:
    median_value = (sorted_vals[mid - 1] + sorted_vals[mid]) / 2

# 4. Mode: count each value in a dictionary, find the highest count
counts = {}
for v in values:
    if v in counts:
        counts[v] += 1
    else:
        counts[v] = 1

highest = 0
for c in counts.values():
    if c > highest:
        highest = c

modes = [v for v, c in counts.items() if c == highest]

print("Mean:", mean_value)
print("Median:", median_value)
print("Mode:", modes)

Mean: 2.4310296735453574
Median: 0.0
Mode: [0.0]


### Data Visualization!

It's not surprising that median and mode are both 0, especially since the database considers a lot of different actions terrorism, not just what we typically think of as terrorism (vandalism, kidnapping, and arson can all be terrorism, for example, but they typically don't include anyone dying). Our mean value does indicate that there are still people dying from terrorism, probably due to some outlier events in the grand scheme of things. Simply tallying up the number of deaths per year could reveal some patterns and trends.

In [37]:
# first I'll add up all the nkill values per year

killed_by_year = tdb2.groupby("iyear")["nkill"].sum()
dataviz = killed_by_year.to_dict()

In [38]:
# now let's show what it calculated. I think a bar graph would probably be
# best for this.

max_value = max(dataviz.values())
BAR_WIDTH = 50  # length of the longest bar, in characters

print("Number of People Killed in Terrorist Attacks per Year")
print()

for year in sorted(dataviz):
    value = dataviz[year]
    length = round(value / max_value * BAR_WIDTH)
    if value > 0 and length == 0:
        length = 1  # keep small nonzero years visible
    bar = "█" * length
    print(f"{int(year)} | {bar} {value:,.0f}")

Number of People Killed in Terrorist Attacks per Year

1970 | █ 174
1971 | █ 173
1972 | █ 485
1973 | █ 370
1974 | █ 539
1975 | █ 617
1976 | █ 674
1977 | █ 457
1978 | ██ 1,459
1979 | ██ 2,100
1980 | █████ 4,391
1981 | █████ 4,851
1982 | ██████ 5,135
1983 | ███████████ 9,444
1984 | ████████████ 10,450
1985 | ████████ 7,094
1986 | ██████ 4,976
1987 | ███████ 6,482
1988 | ████████ 7,206
1989 | █████████ 8,152
1990 | ████████ 7,036
1991 | █████████ 8,454
1992 | ███████████ 9,745
1994 | █████████ 7,690
1995 | ███████ 6,103
1996 | ████████ 6,966
1997 | ████████████ 10,906
1998 | █████ 4,688
1999 | ████ 3,394
2000 | █████ 4,394
2001 | █████████ 7,729
2002 | █████ 4,797
2003 | ████ 3,317
2004 | ██████ 5,716
2005 | ███████ 6,343
2006 | ██████████ 9,316
2007 | ██████████████ 12,825
2008 | ██████████ 9,244
2009 | ██████████ 9,277
2010 | █████████ 7,829
2011 | █████████ 8,246
2012 | █████████████████ 15,498
2013 | █████████████████████████ 22,279
2014 | █████████████████████████████████████████████